In [110]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

In [79]:
df = pd.read_csv("final_dataset.csv")

In [80]:
df_camry = df[(df["model"] == "CAMRY") & (df["year"] > 2019)]
df_camry = df_camry.drop(columns=["detail_url", "make", "model", "photo_urls_json", "make_model_year_price_median"])
df_camry.nunique()

vin                          2681
seller_type                     2
auction                         2
sale_date                     384
sold_price                    454
currency                        2
year                            7
mileage                      2515
engine_volume                   2
cylinders                       2
fuel_type                       4
transmission                    3
drive_type                      3
primary_damage                 24
primary_damage_severity         8
secondary_damage               25
secondary_damage_severity       8
dtype: int64

In [111]:
df_camry.describe(include="all")

,sale_date,seller_type,auction,sold_price,year,mileage,engine_volume,fuel_type,primary_damage,primary_damage_severity,secondary_damage,secondary_damage_severity
count,2648,2648,2648,2648.000000,2648.000000,2545.000000,2646.000000,2648,2648,1998.000000,2648,1622.000000
unique,383,2,2,NaN,NaN,NaN,NaN,3,10,NaN,10,NaN
top,01/15/2026 18:00 UTC,Insurance,copart,NaN,NaN,NaN,NaN,GAS,FRONT END,NaN,NOT_RECORDED,NaN
freq,36,2267,1664,NaN,NaN,NaN,NaN,1729,1372,NaN,876,NaN
mean,NaN,NaN,NaN,9372.598943,2022.609517,59258.088016,2.534769,NaN,NaN,3.756507,NaN,3.358508
std,NaN,NaN,NaN,4068.684903,1.887089,51311.350937,0.183230,NaN,NaN,0.489392,NaN,0.719336
min,NaN,NaN,NaN,100.000000,2020.000000,1.000000,2.500000,NaN,NaN,1.000000,NaN,1.000000
25%,NaN,NaN,NaN,6500.000000,2021.000000,23487.000000,2.500000,NaN,NaN,3.500000,NaN,3.000000
50%,NaN,NaN,NaN,8400.000000,2023.000000,51526.000000,2.500000,NaN,NaN,4.000000,NaN,3.500000
75%,NaN,NaN,NaN,11900.000000,2025.000000,81961.000000,2.500000,NaN,NaN,4.000000,NaN,4.000000


In [112]:
fig_price_dist = px.histogram(
    df_camry,
    x="sold_price",
    nbins=40,
    title="Distribution of Camry Sold Prices",
    labels={"sold_price": "Sold Price (USD)", "count": "Number of Vehicles"},
)
fig_price_dist.show()

In [81]:
df_camry = df_camry.drop_duplicates(subset=["vin"])
df_camry.shape

(2681, 17)

In [82]:
df_camry = df_camry.drop(columns=["vin"])

In [83]:
df_camry.insert(0, 'sale_date', df_camry.pop('sale_date'))

In [108]:
cad_to_usd = 0.713508

df_camry["sold_price"] = df_camry["sold_price"].astype(float)

mask = df_camry["currency"] == "CAD"

df_camry.loc[mask, "sold_price"] = df_camry.loc[mask, "sold_price"] * cad_to_usd

df_camry["sold_price"] = df_camry["sold_price"].astype(int)

df_camry.loc[mask, "currency"] = "USD"

df_camry = df_camry.drop(columns=["currency"])

In [85]:
corr_matrix = corr_data[["engine_volume", "cylinders"]].corr()

fig_heatmap = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation: Engine Volume and Cylinders",
)
fig_heatmap.show()

In [86]:
df_camry = df_camry.drop(columns=["cylinders"])

In [87]:
df_camry.head()

,sale_date,seller_type,auction,sold_price,currency,year,mileage,engine_volume,fuel_type,transmission,drive_type,primary_damage,primary_damage_severity,secondary_damage,secondary_damage_severity
335,03/06/2026 17:00 UTC,Insurance,iaai,6600,USD,2021,76491.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,3.5,FRONT END,4.0
515,03/06/2026 18:00 UTC,Insurance,copart,5500,USD,2023,55536.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,4.0,NaN,NaN
649,03/06/2026 18:00 UTC,Insurance,copart,10500,USD,2020,54505.0,3.5,GAS,AUTOMATIC,FWD,REAR END,4.0,MINOR DENT/SCRATCHES,4.0
997,03/06/2026 15:00 UTC,Insurance,copart,6000,USD,2025,NaN,2.5,HYBRID ENGINE,AUTOMATIC,AWD,REAR END,4.0,SIDE,4.0
1073,03/06/2026 20:00 UTC,Insurance,copart,5500,USD,2022,89059.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,4.0,UNDERCARRIAGE,3.5


In [104]:
df_camry["fuel_type"].value_counts(ascending=False)

fuel_type
GAS              1729
HYBRID ENGINE     576
OTHER             343
Name: count, dtype: int64

In [105]:
df_camry["drive_type"].value_counts(ascending=False)

drive_type
FWD        2414
AWD         232
UNKNOWN       2
Name: count, dtype: int64

In [106]:
df_camry = df_camry.drop(columns=["drive_type"])

In [89]:
df_camry = df_camry[df_camry["fuel_type"] != "UNKNOWN"].copy()

In [90]:
df_camry.groupby("transmission").size().sort_values(ascending=False)

transmission
AUTOMATIC    2648
UNKNOWN        23
MANUAL          1
dtype: int64

In [91]:
df_camry = df_camry[df_camry["transmission"] == "AUTOMATIC"].copy()
df_camry = df_camry.drop(columns=["transmission"])

In [98]:
df_camry["primary_damage_severity"].value_counts(dropna=False)

primary_damage_severity
4.0    1192
NaN     650
3.5     461
3.0     152
4.5     107
2.5      46
2.0      25
1.5      13
1.0       2
Name: count, dtype: int64

In [99]:
df_camry["primary_damage"].value_counts(dropna=False)

primary_damage
FRONT END               1372
REAR END                 545
SIDE                     412
FRONT & REAR              95
ALL OVER                  46
ROLLOVER                  33
HAIL                      32
WATER/FLOOD               21
MINOR DENT/SCRATCHES      20
UNDERCARRIAGE             15
NORMAL WEAR & TEAR        12
MECHANICAL                10
VANDALISM                  6
BURN                       6
ROOF                       5
BIOHAZARD                  5
BURN - ENGINE              3
THEFT                      3
STORM DAMAGE               2
DAMAGE HISTORY             1
REPOSSESSION               1
FRAME DAMAGE               1
SUSPENSION                 1
ENGINE DAMAGE              1
Name: count, dtype: int64

In [100]:
low_count_damage = df_camry["primary_damage"].value_counts()
df_camry.loc[
    df_camry["primary_damage"].isin(low_count_damage[low_count_damage < 20].index),
    "primary_damage"
] = "OTHER"

In [92]:
df_camry["secondary_damage_severity"].value_counts(dropna=False)

secondary_damage_severity
NaN    1026
4.0     544
3.5     508
3.0     255
2.5     135
2.0      71
1.5      63
4.5      31
1.0      15
Name: count, dtype: int64

In [93]:
df_camry["secondary_damage"].value_counts(dropna=False)

secondary_damage
NaN                     876
FRONT END               600
SIDE                    440
REAR END                395
FRONT & REAR             95
MINOR DENT/SCRATCHES     62
UNDERCARRIAGE            50
HAIL                     29
MECHANICAL               20
ROOF                     14
ROLLOVER                 14
ALL OVER                 10
WATER/FLOOD               9
NORMAL WEAR & TEAR        6
BIOHAZARD                 5
BURN                      4
VANDALISM                 4
FRAME DAMAGE              3
THEFT                     3
STORM DAMAGE              2
STRIPPED                  2
REJECTED REPAIR           1
BURN - ENGINE             1
REPOSSESSION              1
SUSPENSION                1
ENGINE DAMAGE             1
Name: count, dtype: int64

In [94]:
pd.crosstab(
    df_camry["secondary_damage"].isna(),
    df_camry["secondary_damage_severity"].isna(),
    rownames=["Damage missing"],
    colnames=["Severity missing"]
)

Severity missing,False,True
Damage missing,,
False,1622,150
True,0,876


In [95]:
df_camry["secondary_damage"] = (df_camry["secondary_damage"].fillna("NOT_RECORDED"))
low_count_damage = df_camry["secondary_damage"].value_counts()
df_camry.loc[
    df_camry["secondary_damage"].isin(low_count_damage[low_count_damage < 20].index),
    "secondary_damage"
] = "OTHER"

In [96]:
df_camry["secondary_damage"].value_counts(dropna=False)

secondary_damage
NOT_RECORDED            876
FRONT END               600
SIDE                    440
REAR END                395
FRONT & REAR             95
OTHER                    81
MINOR DENT/SCRATCHES     62
UNDERCARRIAGE            50
HAIL                     29
MECHANICAL               20
Name: count, dtype: int64

In [109]:
df_camry.nunique()

sale_date                     383
seller_type                     2
auction                         2
sold_price                    456
year                            7
mileage                      2489
engine_volume                   2
fuel_type                       3
primary_damage                 10
primary_damage_severity         8
secondary_damage               10
secondary_damage_severity       8
dtype: int64

In [113]:
df_camry.to_csv("camry_cleaned.csv", index=False)